sauce: https://medium.com/jbennetcodes/how-to-rewrite-your-sql-queries-in-pandas-and-more-149d341fc53e

In [1]:
import requests

In [2]:
target_path = 'data'
urls = [ 'https://ourairports.com/data/airports.csv' \
        ,'https://ourairports.com/data/airport-frequencies.csv' \
        ,'https://ourairports.com/data/runways.csv' \
        ,'https://ourairports.com/data/navaids.csv' \
        ,'https://ourairports.com/data/countries.csv' \
        ,'https://ourairports.com/data/regions.csv' \
       ]
for url in urls:
    response = requests.get(url, stream=True)
    handle = open(target_path + url[url.rindex('/'):], 'wb')
    for chunk in response.iter_content(chunk_size=512):
        if chunk:  # filter out keep-alive new chunks
            handle.write(chunk)

In [3]:
import pandas as pd

airports = pd.read_csv('data/airports.csv')
airport_freq = pd.read_csv('data/airport-frequencies.csv')
runways = pd.read_csv('data/runways.csv')

In [4]:
from pandasql import sqldf

sql = lambda q: sqldf(q, globals())

## SELECT, WHERE, DISTINCT
Here are some SELECT statements. We filter results with WHERE. We use DISTINCT to remove duplicated results.

### Select

#### sql

In [5]:
sql('select * from airports').head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,gps_code,iata_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total Rf Heliport,40.070801,-74.933601,11.0,None,US,US-PA,Bensalem,no,00A,None,00A,None,None,None
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,None,US,US-KS,Leoti,no,00AA,None,00AA,None,None,None
2,6524,00AK,small_airport,Lowell Field,59.949200,-151.695999,450.0,None,US,US-AK,Anchor Point,no,00AK,None,00AK,None,None,None
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,None,US,US-AL,Harvest,no,00AL,None,00AL,None,None,None
4,6526,00AR,closed,Newport Hospital & Clinic Heliport,35.608700,-91.254898,237.0,None,US,US-AR,Newport,no,None,None,None,None,None,00AR


#### df

In [6]:
airports.head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,gps_code,iata_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total Rf Heliport,40.070801,-74.933601,11.0,NaN,US,US-PA,Bensalem,no,00A,NaN,00A,NaN,NaN,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,00AA,NaN,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.949200,-151.695999,450.0,NaN,US,US-AK,Anchor Point,no,00AK,NaN,00AK,NaN,NaN,NaN
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,NaN,US,US-AL,Harvest,no,00AL,NaN,00AL,NaN,NaN,NaN
4,6526,00AR,closed,Newport Hospital & Clinic Heliport,35.608700,-91.254898,237.0,NaN,US,US-AR,Newport,no,NaN,NaN,NaN,NaN,NaN,00AR


### Where

#### sql

In [7]:
sql('select * from airports where ident = "KLAX"')

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,gps_code,iata_code,local_code,home_link,wikipedia_link,keywords
0,3632,KLAX,large_airport,Los Angeles International Airport,33.942501,-118.407997,125.0,None,US,US-CA,Los Angeles,yes,KLAX,LAX,LAX,https://www.flylax.com/,https://en.wikipedia.org/wiki/Los_Angeles_Inte...,None


#### df

In [8]:
airports[airports.ident == 'KLAX']

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,gps_code,iata_code,local_code,home_link,wikipedia_link,keywords
32526,3632,KLAX,large_airport,Los Angeles International Airport,33.942501,-118.407997,125.0,NaN,US,US-CA,Los Angeles,yes,KLAX,LAX,LAX,https://www.flylax.com/,https://en.wikipedia.org/wiki/Los_Angeles_Inte...,NaN


### Distinct

#### sql

In [9]:
sql('select distinct type from airports')

,type
0,heliport
1,small_airport
2,closed
3,seaplane_base
4,balloonport
5,medium_airport
6,large_airport


#### df

In [10]:
airports.type.unique()

array(['heliport', 'small_airport', 'closed', 'seaplane_base',
       'balloonport', 'medium_airport', 'large_airport'], dtype=object)

## SELECT with multiple conditions
We join multiple conditions with an &. If we only want a subset of columns from the table, that subset is applied in another pair of square brackets.

### Select *

#### sql

In [11]:
sql('''
    select * from airports 
    where iso_region = "US-CA" 
    and type = "seaplane_base"
    ''').head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,gps_code,iata_code,local_code,home_link,wikipedia_link,keywords
0,7436,0O0,seaplane_base,San Luis Reservoir Seaplane Base,37.058300,-121.125999,544.0,None,US,US-CA,Los Banos,no,0O0,None,0O0,None,None,None
1,8877,22CA,seaplane_base,Commodore Center Seaplane Base,37.878893,-122.512697,NaN,None,US,US-CA,Mill Valley,no,22CA,None,22CA,None,None,None
2,12298,5CA9,seaplane_base,Konocti - Clear Lake Seaplane Base,38.977699,-122.718002,1326.0,None,US,US-CA,Kelseyville,no,5CA9,None,5CA9,None,None,None
3,16514,C39,seaplane_base,Folsom Lake Seaplane Base,38.707199,-121.133003,466.0,None,US,US-CA,Folsom,no,C39,None,C39,None,None,None
4,16830,CN20,seaplane_base,Ferndale Resort Seaplane Base,39.002998,-122.796997,1326.0,None,US,US-CA,Kelseyville,no,CN20,None,CN20,None,None,None


In [12]:
airports[(airports.iso_region == 'US-CA') \
       & (airports.type == 'seaplane_base')].head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,gps_code,iata_code,local_code,home_link,wikipedia_link,keywords
995,7436,0O0,seaplane_base,San Luis Reservoir Seaplane Base,37.058300,-121.125999,544.0,NaN,US,US-CA,Los Banos,no,0O0,NaN,0O0,NaN,NaN,NaN
2524,8877,22CA,seaplane_base,Commodore Center Seaplane Base,37.878893,-122.512697,NaN,NaN,US,US-CA,Mill Valley,no,22CA,NaN,22CA,NaN,NaN,NaN
6314,12298,5CA9,seaplane_base,Konocti - Clear Lake Seaplane Base,38.977699,-122.718002,1326.0,NaN,US,US-CA,Kelseyville,no,5CA9,NaN,5CA9,NaN,NaN,NaN
14464,16514,C39,seaplane_base,Folsom Lake Seaplane Base,38.707199,-121.133003,466.0,NaN,US,US-CA,Folsom,no,C39,NaN,C39,NaN,NaN,NaN
16881,16830,CN20,seaplane_base,Ferndale Resort Seaplane Base,39.002998,-122.796997,1326.0,NaN,US,US-CA,Kelseyville,no,CN20,NaN,CN20,NaN,NaN,NaN


### Select columns

#### sql

In [13]:
sql('''
    select ident, name, municipality 
    from airports 
    where iso_region = "US-CA" 
    and type = "large_airport"
    ''').head()

,ident,name,municipality
0,KACV,California Redwood Coast-Humboldt County Inter...,Arcata/Eureka
1,KBAB,Beale Air Force Base,Marysville
2,KCMA,Camarillo International Airport,Camarillo
3,KEDW,Edwards Air Force Base,Edwards
4,KFAT,Fresno Yosemite International Airport,Fresno


#### df

In [14]:
airports[(airports.iso_region == 'US-CA') \
       & (airports.type == 'large_airport')]\
          [['ident', 'name', 'municipality']].head()

,ident,name,municipality
30733,KACV,California Redwood Coast-Humboldt County Inter...,Arcata/Eureka
30884,KBAB,Beale Air Force Base,Marysville
31137,KCMA,Camarillo International Airport,Camarillo
31728,KEDW,Edwards Air Force Base,Edwards
31874,KFAT,Fresno Yosemite International Airport,Fresno


## ORDER BY
By default, Pandas will sort things in ascending order. To reverse that, provide ascending=False.

### Asc

#### sql

In [15]:
sql('select * from airport_freq where airport_ident = "KLAX" order by type').head()

,id,airport_ref,airport_ident,type,description,frequency_mhz
0,60767,3632,KLAX,APP,SOCAL APP,36.07
1,60766,3632,KLAX,APP,SOCAL APP,124.30
2,60768,3632,KLAX,ATIS,ATIS,133.80
3,60769,3632,KLAX,CLD,CLNC DEL,121.40
4,60770,3632,KLAX,DEP,SOCAL DEP,124.30


#### df

In [16]:
airport_freq[airport_freq.airport_ident == 'KLAX'].sort_values('type').head()

,id,airport_ref,airport_ident,type,description,frequency_mhz
11958,60767,3632,KLAX,APP,SOCAL APP,36.07
11959,60766,3632,KLAX,APP,SOCAL APP,124.30
11960,60768,3632,KLAX,ATIS,ATIS,133.80
11961,60769,3632,KLAX,CLD,CLNC DEL,121.40
11962,60770,3632,KLAX,DEP,SOCAL DEP,124.30


### Desc

#### sql

In [17]:
sql('select * from airport_freq where airport_ident = "KLAX" order by type desc').head()

,id,airport_ref,airport_ident,type,description,frequency_mhz
0,60776,3632,KLAX,UNIC,UNICOM,122.95
1,60775,3632,KLAX,TWR,TWR,119.80
2,60774,3632,KLAX,OPS,AF,37.22
3,60772,3632,KLAX,MISC,CG,34.50
4,60773,3632,KLAX,MISC,CG,898.40


#### df

In [18]:
airport_freq[airport_freq.airport_ident == 'KLAX'].sort_values('type', ascending=False).head()

,id,airport_ref,airport_ident,type,description,frequency_mhz
11968,60776,3632,KLAX,UNIC,UNICOM,122.95
11967,60775,3632,KLAX,TWR,TWR,119.80
11966,60774,3632,KLAX,OPS,AF,37.22
11964,60772,3632,KLAX,MISC,CG,34.50
11965,60773,3632,KLAX,MISC,CG,898.40


## IN… NOT IN
We know how to filter on a value, but what about a list of values — IN condition? In pandas, **.isin()** operator works the same way. To negate any condition, use ~.

### In

#### sql

In [19]:
sql('select * from airports where type in ("heliport","balloonport")').head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,gps_code,iata_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total Rf Heliport,40.070801,-74.933601,11.0,None,US,US-PA,Bensalem,no,00A,None,00A,None,None,None
1,322658,00CN,heliport,Kitchen Creek Helibase Heliport,32.727374,-116.459742,3350.0,None,US,US-CA,Pine Valley,no,00CN,None,00CN,None,None,None
2,6532,00FD,heliport,Ringhaver Heliport,28.846600,-82.345398,25.0,None,US,US-FL,Riverview,no,00FD,None,00FD,None,None,None
3,6535,00GE,heliport,Caffrey Heliport,33.889245,-84.737930,957.0,None,US,US-GA,Hiram,no,00GE,None,00GE,None,None,None
4,6536,00HI,heliport,Kaupulehu Heliport,19.832715,-155.980233,43.0,None,US,US-HI,Kailua-Kona,no,00HI,None,00HI,None,None,None


#### df

In [20]:
airports[airports.type.isin(['heliport', 'balloonport'])].head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,gps_code,iata_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total Rf Heliport,40.070801,-74.933601,11.0,NaN,US,US-PA,Bensalem,no,00A,NaN,00A,NaN,NaN,NaN
9,322658,00CN,heliport,Kitchen Creek Helibase Heliport,32.727374,-116.459742,3350.0,NaN,US,US-CA,Pine Valley,no,00CN,NaN,00CN,NaN,NaN,NaN
12,6532,00FD,heliport,Ringhaver Heliport,28.846600,-82.345398,25.0,NaN,US,US-FL,Riverview,no,00FD,NaN,00FD,NaN,NaN,NaN
15,6535,00GE,heliport,Caffrey Heliport,33.889245,-84.737930,957.0,NaN,US,US-GA,Hiram,no,00GE,NaN,00GE,NaN,NaN,NaN
16,6536,00HI,heliport,Kaupulehu Heliport,19.832715,-155.980233,43.0,NaN,US,US-HI,Kailua-Kona,no,00HI,NaN,00HI,NaN,NaN,NaN


### Not In

#### sql

In [21]:
sql('select * from airports where type not in ("heliport","balloonport")').head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,gps_code,iata_code,local_code,home_link,wikipedia_link,keywords
0,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,None,US,US-KS,Leoti,no,00AA,None,00AA,None,None,None
1,6524,00AK,small_airport,Lowell Field,59.949200,-151.695999,450.0,None,US,US-AK,Anchor Point,no,00AK,None,00AK,None,None,None
2,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,None,US,US-AL,Harvest,no,00AL,None,00AL,None,None,None
3,6526,00AR,closed,Newport Hospital & Clinic Heliport,35.608700,-91.254898,237.0,None,US,US-AR,Newport,no,None,None,None,None,None,00AR
4,322127,00AS,small_airport,Fulton Airport,34.942803,-97.818019,1100.0,None,US,US-OK,Alex,no,00AS,None,00AS,None,None,None


#### df

In [22]:
airports[~airports.type.isin(['heliport', 'balloonport'])].head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,gps_code,iata_code,local_code,home_link,wikipedia_link,keywords
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,00AA,NaN,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.949200,-151.695999,450.0,NaN,US,US-AK,Anchor Point,no,00AK,NaN,00AK,NaN,NaN,NaN
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,NaN,US,US-AL,Harvest,no,00AL,NaN,00AL,NaN,NaN,NaN
4,6526,00AR,closed,Newport Hospital & Clinic Heliport,35.608700,-91.254898,237.0,NaN,US,US-AR,Newport,no,NaN,NaN,NaN,NaN,NaN,00AR
5,322127,00AS,small_airport,Fulton Airport,34.942803,-97.818019,1100.0,NaN,US,US-OK,Alex,no,00AS,NaN,00AS,NaN,NaN,NaN


## GROUP BY, COUNT, ORDER BY
Grouping is straightforward: use the **.groupby()** operator. There’s a subtle difference between semantics of a **COUNT** in SQL and Pandas. To get the same result as the SQL COUNT, use **.size()**. <div class="alert alert-block alert-info">In Pandas, **.count()** and **groupby()** will return the number of non-null/NaN values.</div> 

### Count, Group By, Order By Asc

#### sql

In [23]:
sql('''
    select iso_country, type, count(*) 
    from airports 
    group by iso_country, type 
    order by iso_country, type
    ''').head(9)

,iso_country,type,count(*)
0,None,closed,4
1,None,large_airport,1
2,None,medium_airport,11
3,None,small_airport,233
4,AD,closed,1
5,AD,heliport,2
6,AE,closed,2
7,AE,heliport,79
8,AE,large_airport,4


#### df

In [24]:
airports.groupby(['iso_country', 'type'], as_index=False).size().head()

,iso_country,type,size
0,AD,closed,1
1,AD,heliport,2
2,AE,closed,2
3,AE,heliport,79
4,AE,large_airport,4


### Order By Desc
Below, we group on more than one field. Pandas will sort things on the same list of fields by default, so there’s no need for a **.sort_values()** in the first example. If we want to use different fields for sorting, or **DESC** instead of **ASC**, like in the second example, we have to be explicit:

#### sql

In [25]:
sql('''
    select iso_country, type, count(*) 
    from airports 
    group by iso_country, type 
    order by iso_country, count(*) desc
    ''').head(9)

,iso_country,type,count(*)
0,None,small_airport,233
1,None,medium_airport,11
2,None,closed,4
3,None,large_airport,1
4,AD,heliport,2
5,AD,closed,1
6,AE,heliport,79
7,AE,small_airport,20
8,AE,medium_airport,7


#### df
What is this trickery with **.to_frame()** and **.reset_index()**? Because we want to sort by our calculated field (**size**), this field needs to become part of the **DataFrame**. After grouping in Pandas, we get back a different type, called a **GroupByObject**. So we need to convert it back to a **DataFrame**. With **.reset_index()**, we restart row numbering for our data frame.

In [26]:
airports.groupby(['iso_country', 'type']).size().to_frame('size')\
.reset_index().sort_values(['iso_country', 'size'], ascending=[True, False]).head()

,iso_country,type,size
1,AD,heliport,2
0,AD,closed,1
3,AE,heliport,79
7,AE,small_airport,20
5,AE,medium_airport,7


## HAVING
In SQL, you can additionally filter grouped data using a **HAVING** condition. In Pandas, you can use **.filter()** and provide a Python function (or a lambda) that will return **True** if the group should be included into the result.

#### sql

In [27]:
sql('''
    select type, count(*) 
    from airports 
    where iso_country = 'US' 
    group by type 
    having count(*) > 1000 
    order by count(*) desc
    ''')

,type,count(*)
0,small_airport,13878
1,heliport,6844
2,closed,3592


#### df

In [28]:
airports[airports.iso_country == 'US'].groupby('type')\
.filter(lambda g: len(g) > 1000).groupby('type')\
.size().sort_values(ascending=False)

type
small_airport    13878
heliport          6844
closed            3592
dtype: int64

#### LIMIT, OFFSET

### Limit

#### sql

In [29]:
sql('select * from airports order by elevation_ft desc limit 5')

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,gps_code,iata_code,local_code,home_link,wikipedia_link,keywords
0,35129,IN-0001,heliport,Siachen Glacier AFS Airport,35.500000,77.000000,22000.0,AS,IN,IN-JK,None,no,None,None,None,None,None,None
1,42716,IN-0003,small_airport,Daulat Beg Oldi Advanced Landing Ground,35.396467,77.928965,16200.0,AS,IN,IN-JK,None,no,None,None,None,None,https://en.wikipedia.org/wiki/Daulat_Beg_Oldi_...,None
2,39635,SPNH,small_airport,Laguna Choclococha Airport,-13.165700,-75.071999,14965.0,SA,PE,PE-HUV,Choclococha,no,SPNH,None,None,None,None,None
3,39624,SPFA,small_airport,Fausa Airport,-14.709400,-71.731102,14809.0,SA,PE,PE-CUS,Fausa,no,SPFA,None,None,None,None,None
4,327348,ZUDC,medium_airport,Daocheng Yading Airport,29.323056,100.053333,14472.0,AS,CN,CN-51,Daocheng County,yes,ZUDC,DCY,None,None,https://en.wikipedia.org/wiki/Daocheng_Yading_...,None


#### df

In [30]:
airports.sort_values('elevation_ft', ascending=False).head(5)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,gps_code,iata_code,local_code,home_link,wikipedia_link,keywords
26514,35129,IN-0001,heliport,Siachen Glacier AFS Airport,35.500000,77.000000,22000.0,AS,IN,IN-JK,NaN,no,NaN,NaN,NaN,NaN,NaN,NaN
26516,42716,IN-0003,small_airport,Daulat Beg Oldi Advanced Landing Ground,35.396467,77.928965,16200.0,AS,IN,IN-JK,NaN,no,NaN,NaN,NaN,NaN,https://en.wikipedia.org/wiki/Daulat_Beg_Oldi_...,NaN
52070,39635,SPNH,small_airport,Laguna Choclococha Airport,-13.165700,-75.071999,14965.0,SA,PE,PE-HUV,Choclococha,no,SPNH,NaN,NaN,NaN,NaN,NaN
52021,39624,SPFA,small_airport,Fausa Airport,-14.709400,-71.731102,14809.0,SA,PE,PE-CUS,Fausa,no,SPFA,NaN,NaN,NaN,NaN,NaN
64626,327348,ZUDC,medium_airport,Daocheng Yading Airport,29.323056,100.053333,14472.0,AS,CN,CN-51,Daocheng County,yes,ZUDC,DCY,NaN,NaN,https://en.wikipedia.org/wiki/Daocheng_Yading_...,NaN


In [31]:
airports.nlargest(5, columns='elevation_ft')

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,gps_code,iata_code,local_code,home_link,wikipedia_link,keywords
26514,35129,IN-0001,heliport,Siachen Glacier AFS Airport,35.500000,77.000000,22000.0,AS,IN,IN-JK,NaN,no,NaN,NaN,NaN,NaN,NaN,NaN
26516,42716,IN-0003,small_airport,Daulat Beg Oldi Advanced Landing Ground,35.396467,77.928965,16200.0,AS,IN,IN-JK,NaN,no,NaN,NaN,NaN,NaN,https://en.wikipedia.org/wiki/Daulat_Beg_Oldi_...,NaN
52070,39635,SPNH,small_airport,Laguna Choclococha Airport,-13.165700,-75.071999,14965.0,SA,PE,PE-HUV,Choclococha,no,SPNH,NaN,NaN,NaN,NaN,NaN
52021,39624,SPFA,small_airport,Fausa Airport,-14.709400,-71.731102,14809.0,SA,PE,PE-CUS,Fausa,no,SPFA,NaN,NaN,NaN,NaN,NaN
64626,327348,ZUDC,medium_airport,Daocheng Yading Airport,29.323056,100.053333,14472.0,AS,CN,CN-51,Daocheng County,yes,ZUDC,DCY,NaN,NaN,https://en.wikipedia.org/wiki/Daocheng_Yading_...,NaN


### Offset

#### sql

In [32]:
sql('select * from airports order by elevation_ft desc limit 5 offset 5')

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,gps_code,iata_code,local_code,home_link,wikipedia_link,keywords
0,28475,SPRF,small_airport,San Rafael Airport,-14.267000,-70.467003,14422.0,SA,PE,PE-PUN,San Rafael,no,SPRF,None,None,None,None,None
1,39554,SLUU,small_airport,Ulla Ulla Airport,-15.033334,-69.250000,14360.0,SA,BO,BO-L,Franz Tamayo,no,SLUU,None,None,None,None,None
2,30788,ZUBD,medium_airport,Qamdo Bangda Airport,30.553600,97.108299,14219.0,AS,CN,CN-54,Bangda,yes,ZUBD,BPX,None,None,https://en.wikipedia.org/wiki/Bangda_Airport,None
3,342197,IN-0279,small_airport,Chushul Airstrip,33.554930,78.722140,14213.0,AS,IN,IN-JK,Chushul,no,None,None,None,None,None,None
4,342200,IN-0282,heliport,Man Heliport,33.857320,78.523980,14127.0,AS,IN,IN-JK,Man,no,None,None,None,None,None,None


In [33]:
airports.nlargest(10, columns='elevation_ft').tail(5)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,gps_code,iata_code,local_code,home_link,wikipedia_link,keywords
52101,28475,SPRF,small_airport,San Rafael Airport,-14.267000,-70.467003,14422.0,SA,PE,PE-PUN,San Rafael,no,SPRF,NaN,NaN,NaN,NaN,NaN
51121,39554,SLUU,small_airport,Ulla Ulla Airport,-15.033334,-69.250000,14360.0,SA,BO,BO-L,Franz Tamayo,no,SLUU,NaN,NaN,NaN,NaN,NaN
64623,30788,ZUBD,medium_airport,Qamdo Bangda Airport,30.553600,97.108299,14219.0,AS,CN,CN-54,Bangda,yes,ZUBD,BPX,NaN,NaN,https://en.wikipedia.org/wiki/Bangda_Airport,NaN
26792,342197,IN-0279,small_airport,Chushul Airstrip,33.554930,78.722140,14213.0,AS,IN,IN-JK,Chushul,no,NaN,NaN,NaN,NaN,NaN,NaN
26795,342200,IN-0282,heliport,Man Heliport,33.857320,78.523980,14127.0,AS,IN,IN-JK,Man,no,NaN,NaN,NaN,NaN,NaN,NaN


## MIN, MAX, MEAN, MEDIAN
<div class="alert alert-block alert-info">SQL does not have median function.</div> To calculate the median in SQL, you will need to use percent_rank(), cume_dist(), ntile(), percentile_disc(), percentile_cont() or other window functions.

In [34]:
runways.head()

,id,airport_ref,airport_ident,length_ft,width_ft,surface,lighted,closed,le_ident,le_latitude_deg,le_longitude_deg,le_elevation_ft,le_heading_degT,le_displaced_threshold_ft,he_ident,he_latitude_deg,he_longitude_deg,he_elevation_ft,he_heading_degT,he_displaced_threshold_ft
0,269408,6523,00A,80.0,80.0,ASPH-G,1,0,H1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,255155,6524,00AK,2500.0,70.0,GRVL,0,0,N,NaN,NaN,NaN,NaN,NaN,S,NaN,NaN,NaN,NaN,NaN
2,254165,6525,00AL,2300.0,200.0,TURF,0,0,01,NaN,NaN,NaN,NaN,NaN,19,NaN,NaN,NaN,NaN,NaN
3,270932,6526,00AR,40.0,40.0,GRASS,0,0,H1,NaN,NaN,NaN,NaN,NaN,H1,NaN,NaN,NaN,NaN,NaN
4,322128,322127,00AS,1450.0,60.0,Turf,0,0,1,NaN,NaN,NaN,NaN,NaN,19,NaN,NaN,NaN,NaN,NaN


### Min

#### sql

In [35]:
sql('select min(length_ft), max(length_ft), count(length_ft), avg(length_ft) from runways')

,min(length_ft),max(length_ft),count(length_ft),avg(length_ft)
0,0.0,120000.0,41833,3263.130184


#### df

In [36]:
runways.agg({'length_ft': ['min', 'max', 'count', 'mean', 'median']}).T

,min,max,count,mean,median
length_ft,0.0,120000.0,41833.0,3263.130184,2738.0


## JOIN
Use **.merge()** to join Pandas dataframes. You need to provide which columns to join on (left_on and right_on), and join type: **inner** (default), **left** (corresponds to LEFT OUTER in SQL), **right** (RIGHT OUTER), or **outer** (FULL OUTER).

#### sql

In [37]:
sql('''
    select airport_ident, airport_freq.type, description, frequency_mhz 
    from airport_freq 
    join airports on airport_freq.airport_ref = airports.id 
    where airports.ident = "KLAX"
    ''').head()

,airport_ident,type,description,frequency_mhz
0,KLAX,APP,SOCAL APP,36.07
1,KLAX,APP,SOCAL APP,124.30
2,KLAX,ATIS,ATIS,133.80
3,KLAX,CLD,CLNC DEL,121.40
4,KLAX,DEP,SOCAL DEP,124.30


#### df

In [38]:
airport_freq.merge(airports[airports.ident == 'KLAX'][['id']], \
                   left_on='airport_ref', right_on='id', how='inner')\
[['airport_ident', 'type', 'description', 'frequency_mhz']].head()

,airport_ident,type,description,frequency_mhz
0,KLAX,APP,SOCAL APP,36.07
1,KLAX,APP,SOCAL APP,124.30
2,KLAX,ATIS,ATIS,133.80
3,KLAX,CLD,CLNC DEL,121.40
4,KLAX,DEP,SOCAL DEP,124.30


## UNION ALL, UNION
Use **pd.concat()** to **UNION ALL** two dataframes. To deduplicate things (equivalent of **UNION**), you’d also have to add **.drop_duplicates()**.

#### sql

In [39]:
sql('''
    select name, municipality 
    from airports 
    where ident = "KLAX" 
    union all 
    select name, municipality 
    from airports 
    where ident = "KLGB"
    ''')

,name,municipality
0,Los Angeles International Airport,Los Angeles
1,Long Beach Airport (Daugherty Field),Long Beach


#### df

In [40]:
pd.concat([airports[airports.ident == 'KLAX'][['name', 'municipality']], \
           airports[airports.ident == 'KLGB'][['name', 'municipality']]])

,name,municipality
32526,Los Angeles International Airport,Los Angeles
32551,Long Beach Airport (Daugherty Field),Long Beach


## INSERT
There’s no such thing as an **INSERT** in Pandas. Instead, you would create a new dataframe containing new records, and then concat the two.

## UPDATE

## DELETE

### Immutability

### Export